# Workshop 3W — Upsell Like an Engineer
Unit 3 — Programming for Intelligent Products

Today we’ll practise TDD (Red → Green → Refactor), experiment logging, and debugging discipline.

In [58]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
import mlflow, pandas as pd
print("MLflow:", mlflow.__version__)
print("Environment ready ✅")

Python: 3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
Platform: Linux-6.8.0-1064-azure-x86_64-with-glibc2.39
MLflow: 3.16.1
Environment ready ✅


In [ ]:
from typing import Literal, TypedDict

class Order(TypedDict):
    time_of_day: str
    loyalty_member: Literal["yes", "no"]
    temperature: int
    order_size: int

# def is_loyal(order: Order) -> bool:
#     return order.get("loyalty_member") == "yes"

def is_hot(order: Order) -> bool:
    return order.get("temperature", 0) >= 30

def is_large(order: Order) -> bool:
    return order.get("order_size", 0) >= 4

def is_loyal_at_lunch(order: Order) -> bool:
    return order.get("time_of_day") == 'lunch' and order.get("loyalty_member") == 'yes'

def is_not_loyal_hot_at_lunch(order: Order) -> bool:
    return order.get("temperature", 0) >= 30 and order.get("loyalty_member") == "no" and order.get("time_of_day") == "lunch"

def should_upsell(order: Order) -> bool:
    """Predict if a customer will buy a drink."""
    # return None  # start with a failing stub (RED)

    # For your current tests, either works because the key is always present. order["loyalty_member"] is useful when the key is required; .get() is useful when it is optional.
    # if (order["time_of_day"] == 'lunch' and order["loyalty_member"] == 'yes'):

    # if (order.get("temperature", 0) >= 30):
    #     return True
    # if (order.get("order_size", 0) >= 4):
    #     return True
    # if (order.get("time_of_day") == 'lunch' and order.get("loyalty_member") == 'yes'):
    #     return True
    # return False

    # if order.get("temperature", 0) >= 30 and order.get("loyalty_member") == "no" and order.get("time_of_day") == "lunch": return False
    # if order.get("loyalty_member") == "yes" and order.get("time_of_day") == "lunch": return True
    # if order.get("temperature", 0) >= 30: return True
    # if order.get("order_size", 0) >= 4: return True
    # return False

    if is_not_loyal_hot_at_lunch(order): return False
    if is_loyal_at_lunch(order) or is_hot(order) or is_large(order): return True
    return False

    # time = str(order.get("time_of_day", "")).strip().lower()
    # loyalty = str(order.get("loyalty_member", "no")).strip().lower()
    # temp = int(order.get("temperature", 0) or 0)
    # size = int(order.get("order_size", 0) or 0)
    # if temp > 30 and time == "lunch" and loyalty != "yes": return False
    # if temp > 30: return True
    # if size >= 4: return True
    # if loyalty == "yes" and time == "lunch": return True
    # return False

In [112]:
# Positive test – loyalty at lunch → True
order_1a: Order = {"time_of_day": "lunch", "loyalty_member": "yes"}
assert should_upsell(order_1a) is True, "FAIL: Loyalty at lunch should be upsold."
print("🟢 PASS: Test 1a")

# Negative test – non-loyalty at lunch → False (prevents `return True`)
order_1b: Order = {"time_of_day": "lunch", "loyalty_member": "no"}
assert should_upsell(order_1b) is False, "🔴 FAIL: Non-loyalty at lunch should NOT be upsold."
print("🟢 PASS: Test 1b")

# Evening loyalty → should NOT be upsold
order_2: Order = {"time_of_day": "evening", "loyalty_member": "yes"}
assert should_upsell(order_2) is False, "🔴 FAIL: Evening loyalty should NOT be upsold."
print("🟢 PASS: Test 2")

🟢 PASS: Test 1a
🟢 PASS: Test 1b
🟢 PASS: Test 2


In [61]:
try:
    import mlflow
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("hotdog-upsell")
    with mlflow.start_run(run_name="Is loyal at lunch"):
        order: Order = {"time_of_day": "lunch", "loyalty_member": "no"}
        prediction = should_upsell(order)
        mlflow.log_params(order)
        mlflow.log_metric("prediction", int(bool(prediction)))
        mlflow.set_tag("rule_version", "v1")
    with mlflow.start_run(run_name="Is hot"):
            order: Order = {"temperature": 32}
            prediction = should_upsell(order)
            mlflow.log_params(order)
            mlflow.log_metric("prediction", int(bool(prediction)))
            mlflow.set_tag("rule_version", "v1")
    with mlflow.start_run(run_name="Is large"):
                order: Order = {"order_size": 5}
                prediction = should_upsell(order)
                mlflow.log_params(order)
                mlflow.log_metric("prediction", int(bool(prediction)))
                mlflow.set_tag("rule_version", "v1")
    print("Experiment logged successfully ✅")
except Exception:
    print("[INFO] MLflow not installed — skipping logging safely.")


Experiment logged successfully ✅


In [105]:
# Heatwave (temperature 30 or greater) → True
order_3 = {"time_of_day": "evening", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_3) is True, "🔴 FAIL: Hot weather should trigger upsell."
print("🟢 PASS: Test 3")

# Large order (order size 4 or greater) → True
order_4 = {"order_size": 4}
assert should_upsell(order_4) is True, "🔴 FAIL: Large orders should trigger upsell."
print("🟢 PASS: Test 4")

🟢 PASS: Test 3
🟢 PASS: Test 4


In [106]:
# Busy lunch heatwave (non-loyalty) → should NOT upsell
order_5 = {"time_of_day": "lunch", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_5) is False, "🔴 FAIL: Busy lunch heatwave (non-loyalty) should NOT upsell."
print("🟢 PASS: Test 5")

🟢 PASS: Test 5


### 🎓 Wrap-up
- 🤔 How did TDD (🔴 red → 🟢 green → 🔄 refactor) and small commits reinforce each other in this workshop?
- 🤔 Which TDD habit will you keep using first when you start a new feature?
- 🤔 After this intro to Git, what's one small Git habit you'll practice next time (UI or CLI)?

### 🚀 Going further
- Move on to Activity 7 to bridge these habits into a small ML pipeline (see `activities/activity-7/activity-7__start.md`), using the same TDD + defensive + observability mindset.